# Predictive Analysis — Falcon 9 Landing Classification

**Goal:** predict the binary landing outcome using launch-site, orbit, payload, chronology and flight information.

Models compared: Logistic Regression, Support Vector Machine, Decision Tree and K-Nearest Neighbors.

In [1]:
from pathlib import Path
def resolve_data(filename):
    candidates = [Path("data")/filename, Path("../data")/filename]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(filename)

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

df=pd.read_csv(resolve_data("spacex_launch_data.csv"))
X=pd.get_dummies(df[["FlightNumber","LaunchSite","Orbit","PayloadMass","Year"]], drop_first=False)
y=df["Class"]
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=.30,random_state=42,stratify=y
)
print("Training records:",len(X_train))
print("Test records:",len(X_test))


Training records: 14
Test records: 6


In [2]:

models={
    "Logistic Regression":Pipeline([("scale",StandardScaler()),("model",LogisticRegression(max_iter=2000))]),
    "SVM":Pipeline([("scale",StandardScaler()),("model",SVC())]),
    "Decision Tree":DecisionTreeClassifier(max_depth=4,random_state=42),
    "KNN":Pipeline([("scale",StandardScaler()),("model",KNeighborsClassifier(n_neighbors=3))])
}
results={}
predictions={}
for name,model in models.items():
    model.fit(X_train,y_train)
    pred=model.predict(X_test)
    predictions[name]=pred
    results[name]=accuracy_score(y_test,pred)

scores=pd.Series(results,name="Test Accuracy").sort_values(ascending=False)
scores


Logistic Regression    0.833333
SVM                    0.833333
KNN                    0.833333
Decision Tree          0.666667
Name: Test Accuracy, dtype: float64

In [3]:

best_name=scores.index[0]
print("Best model:",best_name)
print("Test accuracy:",scores.iloc[0])
print("Confusion matrix:")
print(confusion_matrix(y_test,predictions[best_name]))


Best model: Logistic Regression
Test accuracy: 0.8333333333333334
Confusion matrix:
[[0 1]
 [0 5]]


## Evaluation

The test-set score and confusion matrix are evaluated together. Because this repository includes a compact reproducible dataset, the model result is demonstrative; the original coursework result should be preserved in the final presentation where applicable.